# Victorian Fire — Weather Data Collection & Enrichment
**Project:** FireFusion | AI Modelling Stream  
**Purpose:** Fetch historical weather data from Open-Meteo and enrich fire events with weather variables at ignition date and location  
**Sections:**
1. Config & Imports
2. Fire Data Loading
3. Weather Enrichment (Open-Meteo Historical Archive)
4. Export

## 1. Config & Imports

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# ── File paths ────────────────────────────────────────────────────────────
RAW_FILE    = "../../src/data/fire_history_2019_2023.geojson"
JOINED_FILE = "../../src/data/fire_weather_joined.geojson"

# ── Weather variables from Open-Meteo ──────────────────────────────────────
WEATHER_VARS = [
    "temperature_2m_max",
    "relative_humidity_2m_max",
    "wind_speed_10m_max",
    "wind_direction_10m_dominant",
    "precipitation_sum",
    "et0_fao_evapotranspiration",
]

RENAME_MAP = {
    "temperature_2m_max":            "max_temp_c",
    "relative_humidity_2m_max":       "rel_humidity_pct",
    "wind_speed_10m_max":             "wind_speed_kmh",
    "wind_direction_10m_dominant":    "wind_dir_deg",
    "precipitation_sum":              "precipitation_mm",
    "et0_fao_evapotranspiration":     "evapotranspiration",
}

print("Config loaded.")

Config loaded.


## 2. Fire Data Loading
Load preprocessed fire history data (from victorian_fire_history.ipynb)

In [2]:
# Load the fire history geojson
fire_data = gpd.read_file(RAW_FILE)

# Type casting
fire_data["season"]     = fire_data["season"].astype(str)
fire_data["area_ha"]    = pd.to_numeric(fire_data["area_ha"], errors="coerce")
fire_data["start_date"] = pd.to_datetime(fire_data["start_date"], utc=True, errors="coerce")

# Filter to bushfires only
bushfires = fire_data[fire_data["firetype"] == "Bushfire"].copy()
bushfires = bushfires[bushfires["area_ha"] > 0].copy()

# Centroid calculation
centroids_wgs = (
    bushfires.to_crs(epsg=7855)
             .geometry
             .centroid
             .pipe(lambda s: gpd.GeoSeries(s, crs="EPSG:7855"))
             .to_crs(epsg=4326)
)
bushfires = bushfires.to_crs(epsg=4326)
bushfires["lon"]      = centroids_wgs.x
bushfires["lat"]      = centroids_wgs.y
bushfires["date_str"] = bushfires["start_date"].dt.strftime("%Y-%m-%d")

print(f"Loaded {len(bushfires):,} bushfire records")
print(bushfires[["season", "date_str", "area_ha", "lat", "lon"]].head())

Loaded 68,738 bushfire records
    season    date_str     area_ha        lat         lon
4     2019  2019-01-29    2.065290 -38.861292  146.373019
5     2019  2019-03-01  318.069812 -38.996923  146.315309
6     2019  2018-11-18   59.677880 -38.597696  143.322586
7     2019  2019-02-04    1.493813 -38.379332  141.641153
661   2019  2019-02-01  269.943595 -38.433491  145.523179


## 3. Weather Enrichment (Open-Meteo Historical Archive)
For each bushfire event, fetch daily weather from the fire ignition date and location.  
**Note:** Change `SAMPLE_N` to `None` to run on the full dataset.

In [ ]:
def fetch_weather(lat: float, lon: float, date_str: str) -> dict:
    """Fetch daily weather for a single location and date from Open-Meteo archive."""
    try:
        resp = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat, "longitude": lon,
                "start_date": date_str, "end_date": date_str,
                "daily": WEATHER_VARS,
                "wind_speed_unit": "kmh",
                "timezone": "Australia/Melbourne",
            },
            timeout=15,
        )
        resp.raise_for_status()
        daily = resp.json().get("daily", {})
        if not daily or not daily.get("time"):
            return {}
        return {RENAME_MAP[k]: daily[k][0] for k in WEATHER_VARS if k in daily}
    except Exception as e:
        print(f"  Weather fetch failed ({lat:.3f}, {lon:.3f}, {date_str}): {e}")
        return {}


# ── Configuration ─────────────────────────────────────────────────────────
SAMPLE_N = 20  # set to None for full dataset

target = bushfires.head(SAMPLE_N) if SAMPLE_N else bushfires
empty_weather = {v: None for v in RENAME_MAP.values()}

weather_rows = []
for i, (idx, row) in enumerate(target.iterrows(), 1):
    print(f"[{i}/{len(target)}] {row['date_str']}  ({row['lat']:.3f}, {row['lon']:.3f})")
    w = fetch_weather(row["lat"], row["lon"], row["date_str"])
    weather_rows.append({**empty_weather, **w, "_idx": idx})
    time.sleep(0.2)  # Rate limiting

weather_df = pd.DataFrame(weather_rows).set_index("_idx")
joined = target.join(weather_df)

print(f"\nJoined shape: {joined.shape}")
print(joined[[
    "season", "date_str", "area_ha",
    "max_temp_c", "rel_humidity_pct", "wind_speed_kmh", "wind_dir_deg"
]].head(10).to_string())

## 4. Export
Save weather-enriched fire dataset in multiple formats.

In [ ]:
# ── Save joined dataset ────────────────────────────────────────────────────
joined.drop(columns="geometry").to_csv("../../src/data/fire_weather_joined.csv", index=False)
joined.to_file(JOINED_FILE, driver="GeoJSON")

# ── Summary stats ──────────────────────────────────────────────────────────
print("=" * 55)
print("WEATHER ENRICHMENT SUMMARY")
print("=" * 55)
print(f"Total bushfire records : {len(bushfires):>8,}")
print(f"Weather-joined records : {len(joined):>8,}")
print(f"Weather null rate      : {joined['max_temp_c'].isna().mean():>8.1%}")
print("-" * 55)
print(f"Outputs:")
print(f"  {JOINED_FILE}")
print(f"  ../../src/data/fire_weather_joined.csv")